# Introduction to Xarray for Working with Labeled Numerical Data Arrays

<div class="alert alert-success">
    
## This notebook covers
- Xarray data structures
- Indexing/selecting
- Summary statistics, masks, searching, sorting
- Groupby and resample
- Spatial and temporal weighting
- Assert testing functions
- Array creation
- NumPy/Pandas/Xarray data structure conversions
- Reading and writing array data
- Creating figures with Matplotlib and Cartopy
</div>

<div class="alert alert-warning">

## Reminders

Remember, you can use Jupyter's built-in table of contents (hamburger on the far left) to jump from heading to heading.

---

This notebook will run in the MSUpy conda environment, which you created in a previous lesson. To load the MSUpy environment in this notebook go to the Kernel tab, select Change Kernel, then select the MSUpy kernel in the pop up window.

---

To turn on line numbers for code cells go to View menu and click Show Line Numbers.

</div>

# I. Importing Necessary Packages

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import copy

# II. Introduction to the Xarray Data Structures

Xarray builds upon many other Python packages, including Numpy, Pandas, Scipy, netCDF4, Matplotlib, and more. The Xarray package is most convenient when working with multi-dimensional data stored in netcdf files. Xarray can also handle zarr, tiff, csv, hdf, and grib files but may require additional dependencies to be installed, more info can be found in the [Xarray FAQs](https://docs.xarray.dev/en/latest/getting-started-guide/faq.html#how-do-i-open-format-x-file-as-an-xarray-dataset). 

From the [Xarray documentation](https://docs.xarray.dev/en/stable): "Xarray introduces labels in the form of dimensions, coordinates, and attributes on top of raw Numpy-like multidimensional arrays, which allows for a more intuitive, more concise, and less error-prone developer experience".

We'll cover what all that means shortly! Another gigantic benefit of Xarray is that it integrates well with the Dask package for parallel and distributed computing, which enables fast computation on large data. This aspect of Xarray is a bit too advanced for this course, but it's worth mentioning here, regardless. 

Xarray is under active community development and pushes new updates approximately monthly. This means that developers are actively working on improvements and expanded capabilities and that there will probably be useful updates more frequently than you may be used to.

## Data Structures - DataArray and Dataset

Xarray's core data structures are called the *DataArray* and the *Dataset*. A DataArray is an n-dimensional array of a single data variable with *labels* (*metadata*) that describe the *dimensions*, *coordinates*, and *attributes* of the data. A Dataset contains one or more DataArrays which share one or more dimensions and coordinates. We will cover what all this Xarray terminology means below, but you can also find more detail on the [Xarray User Guide Terminology page](https://docs.xarray.dev/en/stable/user-guide/terminology.html).

Let's look at an Xarray Dataset object and walk through all its components. We'll load data from the file ```data/nclimgrid/nclimgrid_tmax_199401-202312.nc```. This netcdf file contains the monthly averages of daily maximum surface air temperature (tmax) for 30 years on a spatial grid. The original source of this data is [NOAA Monthly U.S. Climate Gridded Dataset (NClimGrid)](https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332) but the file we are working with has been subset in time and clipped to the state of Mississippi.

We can use the function ```xr.open_dataset()``` to easily load data from the netcdf file into an Xarray Dataset object. As you can see below, when we print the Dataset object ```ds``` to the screen, we get a ton of information. This information is what was meant above by the terms metadata and labels.

In [ ]:
# read contents of netcdf file into a Dataset object
ds = xr.open_dataset('data/nclimgrid/nclimgrid_tmax_199401-202312.nc')
ds

A Dataset object will contain one or more DataArrays, which are listed under the Data variables section of the info printed to the screen. Our Dataset object ```ds``` contains one DataArray called ```tmax```. 

Click on the paper icon to the right of the DataArray ```tmax``` and you will see even more metadata labels. These are called *variable attributes*. Variable attributes are contained in a Python dictionary where the dictionary keys are the attribute names (units, standard_name, etc.) and the dictionary values are the attribute values (e.g., degree_Celsius, air_temperature). If we were writing our own data to a netcdf file, we could include any variable attributes we want to help describe the data. But generally when you are working with climate data, the convention is to use the Climate and Forecast (CF) Metadata Conventions.

<div class="alert alert-danger">

**Sidebar: [Climate and Forecast (CF) Metadata Conventions](https://cfconventions.org)** 

The CF Conventions are essentially a set of rules for how climate data should be described and written to data files in order to promote standardized data processing, eliminate ambiguities, and facilitate data sharing. The data file we are working with uses the CF Conventions and that is why the ```tmax``` variable attributes have those specific names (e.g., standard_name, valid_min)- they come from the list of attributes in the [CF Metadata Conventions Appendix A: Attributes](https://cfconventions.org/cf-conventions/cf-conventions.html#attribute-appendix).  
</div>

The data file itself may also have attributes called *file attributes* or *global attributes* that are separate from variable attributes. We can find these attributes in the print out above under the Attributes section. This particular file doesn't have any file attributes as indicated by the zero next to the Attributes section. You can imagine, though, that if you were to write your own netcdf file of data, you may want to include file attributes like "institution" or "Conventions" (these examples come from Appendix A of the CF Conventions linked above in the sidebar) to indicate inside your data file what institution created that file and what version of the CF Conventions was used.

Other information we can see about the ```tmax``` DataArray includes the data type (float32) and the dimension names and order (time,lat,lon). Unlike NumPy arrays, where axis 0, axis 1, axis 2, etc can represent anything, there is no ambiguity about what each dimension represents with Xarray data structures because they are labeled. If we look toward the top of the print out we can see each dimension length. The ```tmax``` DataArray, which is stored inside the ```ds``` Dataset, contains temperature data for 360 times at 116 latitudes and 85 longitudes.   

Each of our three dimensions is associated with a *coordinate variable*. Coordinate variables (or simply "coordinates") enable us to include very useful metadata about the dimensions of our DataArrays. Click the data stack icon next to any of the coordinates in the print out above and we see the time, latitude, and longitude values that are associated with the data variables. Each value of a coordinate is indexed along one dimension (either time, latitude, or longitude) to a position in the data variable ```tmax```. For example, we see that the time coordinate begins in January 1994 and ends in December 2023, and since time is a coordinate of ```tmax```, we know that these times match the indexes of the ```tmax``` time dimension. Click the paper icon next to the time coordinate and we see that each coordinate variable also has its own attributes! This is because coordinates are also DataArray structures. Coordinate variables are 1-dimensional DataArrays that hold data dimension values and dimension attributes. Coordinates are super important as they allow us to select data using labels instead of index positions as we will see shortly.   

Now let's pull the ```tmax``` DataArray out of the Dataset into a new variable in our notebook called ```tx```. We can access the variables in a dataset using a dot and the data variable name.

In [ ]:
# get data array structure from dataset structure
tx = ds.tmax
tx

The print out tells us that our new variable ```tx``` is an Xarray DataArray object. Notice that all of the variable attributes and coordinates that we saw associated with ```tmax``` inside the Dataset are still attached.

An alternative syntax for the above looks a lot like how we accessed a single Series from a Pandas DataFrame.

In [ ]:
# get data array structure from dataset structure
tx = ds['tmax']
tx

We can get a dictionary-like object of the data variables within a Dataset object using the Dataset property ```.data_vars```

In [ ]:
# get data array info from dataset structure
ds.data_vars

To get a list of only the variable names in a dataset we can use Python's built-in ```list()``` function.

In [ ]:
# get list of data array names inside of dataset structure
list(ds.data_vars)

This may be helpful if you need to automate the processing (using a loop) of multiple data variables within a Dataset. You could use the code above to get a list of variable names and then use those variable names one by one to access different data variables inside the Dataset.

## Accessing the Components of DataArray Objects

As we saw above, the main components in a DataArray structure are dimensions, coordinates, attributes, and the array of data values. We can access these components using various DataArray properties.
- ```.name``` returns the string name of the DataArray as it was named in the netcdf file
- ```.dims``` returns a tuple of dimension names
- ```.sizes``` returns a dictionary of dimension names and lengths
- ```.coords``` returns a dictionary-like object of coordinate information
- ```.attrs``` returns a dictionary of variable attributes
- ```.data``` returns the underlying NumPy array of data values 

In [ ]:
# get DataArray name string
tx.name

In [ ]:
# get tuple of dimension names
tx.dims

In [ ]:
# get dictionary of dimension info
tx.sizes

In [ ]:
# get dictionary of coordinate info
tx.coords

In [ ]:
# get dictionary variable attributes
tx.attrs

To access only the data values in a DataArray without all the additional metadata attached we can use the ```.data``` property. Run the code cell below to see that the underlying structure holding the data values is a NumPy ndarray! We can think of the ```.data``` property as a conversion from an Xarray data structure to a NumPy data structure.

In [ ]:
# get the underlying NumPy array of data values
print(type(tx.data))
tx.data

Inside a DataArray structure, attributes are attached to the data and coordinate variables and all the coordinate information is indexed to the data variable. This means **we can use coordinate names and attribute names to access specific coordinate and attribute values directly from the data variable.** 

For example, we can access the DataArray object for a specific coordinate by using a dot and the coordinate name on our data variable.

In [ ]:
# get a specific coordinate DataArray object
tx.lat

To access the attributes of a coordinate we can use the ```.attrs``` property on a coordinate DataArray object.

In [ ]:
# get coordinate attributes
tx.lat.attrs

Dictionary syntax will get us the attribute value paired to a particular attribute key.

In [ ]:
# get single attribute value
tx.lat.attrs['standard_name']

Of course this syntax also works for accessing any of the variable attributes attached to ```tmax```.

In [ ]:
# get a specific variable attribute
tx.attrs['units']

Quick review of the syntax we've used and what type of object is returned:
- ```ds``` is a Dataset object
- ```tx = ds.tmax``` is a DataArray object 
- ```tx.data``` is a NumPy ndarray object
- ```tx.lat``` is a DataArray object
- ```tx.lat.attrs``` is a dictionary
- ```tx.lat.attrs['standard_name']``` is a string

Before we move on, you may have noticed that there is an Indexes section in the print out of each DataArray and Dataset we've looked at. Indexes are associated with coordinate variables. Coordinate variables are special in that their data values are held in two underlying data structures: a NumPy array and a Pandas Index. This is so the Xarray package can build functionality on top of both the NumPy and Pandas packages. The coordinate variable indexes in a DataArray enable fast label-based indexing as we'll see next. These indexes are working behind the scenes and we don't have to worry about accessing these indexes directly.

You are probably realizing by now that the Xarray data structures are very complex as compared to the other data structures we've learned about thus far in the course. But this complexity will actually make data analysis a lot simpler and less error prone.

## Properties of Xarray DataArrays that Come from NumPy

The Xarray package incorporates much of the same functionality for its DataArray structure that NumPy has for its ndarray data structure. Here are some of the array properties we covered in the NumPy lesson that are also available with Xarray. These Xarray properties provide information about only the underlying NumPy array of data inside an Xarray DataArray structure. Documentation can be found in the [Xarray API Reference](https://docs.xarray.dev/en/latest/api.html#ndarray-attributes).

- ```DataArray.shape```, tuple of dimension lengths
- ```DataArray.ndim```, number of dimensions
- ```DataArray.size```, total number of elements
- ```DataArray.dtype```, data type
- ```DataArray.nbytes```, total bytes consumed by the NumPy array

In [ ]:
# tx is a DataArray
# get information about the underlying NumPy Array of data

print(tx.shape)   # tuple of dimension lengths
print(tx.ndim)    # number of dimensions
print(tx.size)    # total number of elements (360*116*85)
print(tx.dtype)   # data type
print(tx.nbytes)  # total bytes consumed by the NumPy array

In [ ]:
# the coordinate tmax.time is a DataArray
# get information about the underlying NumPy Array of data

print(tx.time.shape)   # tuple of dimension lengths
print(tx.time.ndim)    # number of dimensions
print(tx.time.size)    # total number of elements 
print(tx.time.dtype)   # data type
print(tx.time.nbytes)  # total bytes consumed by the NumPy array

## Data Types and Conversion

Netcdf files can contain binary, numeric, and string data, so Xarray is built to handle all these data types as well. When reading data from a netcdf file with Xarray, a lot happens behind the scenes as the file contents are divvied up into the different components of an Xarray data structure. Climate data variables and their coordinates from a netcdf file are usually interpreted as NumPy numeric or datetime data types when they are read from a file into the NumPy arrays that underlay the Xarray DataArray structure. Labels and attributes from a file are read into strings, tuples of strings, and dictionaries of strings as we've already seen. Xarray does not have its own special data types like we saw with NumPy. Instead, Xarray incorporates the data types that NumPy offers. 

Xarray has implemented the ```.astype()``` function from NumPy for Xarray DataArrays so we can easily convert data types just as we did with NumPy. The same potential pitfalls about unsafe data type conversions apply with data in Xarray structures just as they did with NumPy. We won't cover that again here, but look back to the NumPy lesson if you need a refresher.

Let's convert ```tx``` from data type float32 to type float16.

In [ ]:
# convert data type of tx
tx = tx.astype(np.float16)
tx

Notice how the data type of the ```tx``` array changed but the data types of its coordinates remained the same. The temperature data and the coordinates that are indexed to that data are all stored in separate NumPy arrays within the Xarray DataArray structure. This is why changing the data type of one underlying NumPy array will not affect the type of any other underlying NumPy array within a DataArray object. We could also change the data type of a coordinate if we wanted to.

In [ ]:
# convert data type of a coordinate
tx.coords['lon'] = tx.lon.astype(np.float64)
tx

<div class="alert alert-info"> 

## Exercise 1: Getting Familiar with the Components of a DataArray

Use the variable ```tx``` that we've already created to complete the exercise.

A) In the ```ds``` Dataset shown in the image below, what is the Xarray terminology for the text in the purple box?

<img src="images/xarray_ex1.png" alt="dataset print out" width="400"/>

</div>

Type your answer here: 

<div class="alert alert-info"> 

B) What type of object will ```tx.time``` return?
</div>

Type your answer here: 

<div class="alert alert-info"> 

C) What type of object will ```tx.time.data``` return?
</div>

Type your answer here: 

<div class="alert alert-info"> 

D) What type of object will ```tx.time.attrs``` return?
</div>

Type your answer here: 

<div class="alert alert-info"> 

E) Save the ```tx``` longitude coordinate to a new DataArray variable called ```lon```.
</div>

In [ ]:
# add your code here


<div class="alert alert-info"> 

F) Print the value of the ```tx``` variable attribute ```long_name```.
</div>

In [ ]:
# add your code here


<div class="alert alert-info"> 

G) Print the value of the ```tx``` time coordinate attribute ```long_name```.
</div>

In [ ]:
# add your code here


<div class="alert alert-info"> 

H) Convert the values of the ```tx``` time coordinate to a NumPy ndarray and save them to a new variable called ```time```. Use the ```type()``` function to verify that your ```time``` variable is a NumPy ndarray.
</div>

In [ ]:
# add your code here


# III. Indexing and Slicing

Let's look at how the dimension labels and coordinates of a DataArray allow us to use label-based indexing and slicing.

## Label-based Selection with .sel()

The DataArray function [```.sel()```](https://docs.xarray.dev/en/latest/generated/xarray.DataArray.sel.html#xarray.DataArray.sel) allows us to use dimension and coordinate labels to select parts of a data variable.

### Indexing

Let's select temperature data at a single time.

In [ ]:
# select data at a single time
tx.sel(time='2000-01-01')

Notice what was returned. We recieved a 2-dimensional DataArray: all the latitudes and longitudes of data for January 2000. If we think of our data as spatial maps of temperature arranged in a stack, where each map in the stack represents a different point in time, then we just selected a single map from the stack. This is similar to what we did in the NumPy lesson, except this time we have labels to make this selection much more clearly. We don't have to know which axis (0, 1, or 2) represents time because now there is a dimension label 'time'. And we don't have to figure out what index along the time dimension represents January 2000 because there is a coordinate label for that. Notice that when we select a single time, we do not get a singleton dimension (time dimension dissapears), but we do retain the coordinate label in case we need it later- very convenient. 

We can provide more dimension labels and coordinate values to ```.sel()``` if we want to select data in multiple dimensions.

In [ ]:
# select data at a single time, a single latitude, and all longitudes 
tx.sel(time='2000-01-01',lat=30.3542)

In [ ]:
# select a single data value corresponding to a time, lat, and lon
tx.sel(time='2000-01-01',lat=30.3542,lon=-88.6875)

By the way, if you wanted to extract a single number from an Xarray object as a Python scalar object (without any of the metadata attached) you can use the Xarray function ```.item()```.

In [ ]:
tx.sel(time='2000-01-01',lat=30.3542,lon=-88.6875).item()

If we give ```.sel()``` an inexact latitude and longitude and also provide the parameter ```method="nearest"``` we can get the nearest data point in space.

In [ ]:
# get the nearest point to an inexact lat and lon
tx.sel(time='2000-01-01',lat=30.3,lon=-88.6, method="nearest")

We can select with a list of labels as well. This is helpful, for example, if we want to select multiple times that aren't consecutive.

In [ ]:
# select multiple times that are not consecutive
tx.sel(time=['2000-01','2000-04','2000-07','2000-10'])

The list of labels we use for selecting can be in any order. The data will be returned in the same order as the label list.

In [ ]:
# select multiple times that are not consecutive in whatever order
tx.sel(time=['2000-01','2000-10','2000-07','2000-04'])

Want to select all data for a single year? Easy! (Because our times are datetimes objects) 

In [ ]:
# select one full year
tx.sel(time='2000')

Xarray also enables indexing using datetime components with the ```.dt``` accessor. This makes time selection super convenient. The following examples use ```.dt.month``` but selection will work with any of the ```.dt``` accessor fields. See the full list in the Xarray documentation for [xarray.core.accessor_dt.DatetimeAccessor](https://docs.xarray.dev/en/latest/generated/xarray.core.accessor_dt.DatetimeAccessor.html).

In [ ]:
# select all the Junes
tx.sel(time=(tx.time.dt.month==6))

In [ ]:
# select all Oct,Nov,Dec
tx.sel(time=tx.time.dt.month.isin([10,11,12]))

### Slicing

We can also slice data using labels. Notice that just like we saw with Pandas DataFrames in the Pandas lesson, label-based slicing with Xarray is inclusive of the ending label.

Let's select all data for 6 months in time.

In [ ]:
# a slice of 6 months
tx.sel(time=slice('2000-02','2000-07'))

Let's continue slicing... we can slice as many dimensions as we want.

In [ ]:
# a slice of every dimension
tx.sel(time=slice('2000-02','2000-07'), lat=slice(30.3542,30.6042), lon=slice(-88.9375,-88.6875))

We can also combine label-based indexing and slicing together.

In [ ]:
# one time, slice of lats and lons
tx.sel(time='2020-01',lat=slice(30.3542,30.6042), lon=slice(-88.9,-88.6))

Exact matches to the coordinate values are not required for slicing. With inexact slice values like below, the latitude slice we'll get will be 30.0 <= lat <= 30.5 and the longitude slice will be -89.0 <= lon <= -88.5. Xarray will select all latitudes and longitudes that meet those criteria.

In [ ]:
# slice with inexact latitude, longitude values 
tx.sel(time='2020-01',lat=slice(30.,30.5), lon=slice(-89.0,-88.5))

We can also select with an array of coordinate values.

In [ ]:
# create array of datetimes
time_arr = tx.time.sel(time=slice('2000','2002'))

# select temperature data using an array of times
tx.sel(time=time_arr)

### When Order Does and Does Not Matter with Label-based Selection

An important thing to note is that the **order of the dimension names inside ```.sel()``` is irrelevant**. For example, our ```tx``` array dimensions are ordered (time, lat, lon) but as long as we are using labels we can make selections with the dimension names in any order. This feature, in particular, can help you avoid a lot of coding mistakes.

In [ ]:
# order of dimension names in .sel() is irrelevant
tx.sel(lon=slice(-89.0,-88.5),lat=slice(30.,30.5),time='2020-01')

However, **order is important inside of the ```slice()``` function**. The slice needs to be ordered from left to right according to the order of the coordinate values. For example, because our latitudes are ordered ascending (30.187532 to 34.9792) we could not do ```slice(33,32)```. This applies to slices based on any coordinate. We may not receive an error immediately, but we will see in the print out of the metadata that zero latitudes are selected and that the data array has no contents, which will eventually throw an error in your code.

In [ ]:
# slicing incorrectly for latitudes that are ordered ascending
tx.sel(lat=slice(33,32))

If the latitude coordinate in our data was descending instead of ascending (which you will find is the case with plenty of data), the values you put into ```slice()``` would still go from left to right, meaning the larger latitude would come first, followed by the smaller latitude like we did above. Although, it's probably a better idea just to reorder all your coordinates so that they are all ascending to reduce confusion.

Let's reorder latitude descending so we can see how this works. To reverse the order of your data along a coordinate you can use Xarray's ```.reindex()``` function in combination with some syntax we learned way back in the Python Language Basics lesson ```[::-1]```.

In [ ]:
# reverse order or latitudes
tx_reordered = tx.reindex(lat=tx.lat[::-1])
tx_reordered

Now, we can successfully make the same selection we tried a few notebook cells up, because now ```slice(33,32)``` matches the values of the latitude coordinate from left to right.

In [ ]:
# order of values inside slice() go from left to right according to how latitudes are ordered
tx_reordered.sel(lat=slice(33,32))

For clarity, when we use ```.reindex()``` not only is the coordinate reordered, but the temperature data values are also reordered. This is because each coordinate is an index to the data values. This means that each value of a coordinate references a particular position in the ```tx``` data array. You can think of the coordinate values as being attached to specific data values such that when you reorder a coordinate, the data values get reordered as well. This is a very convenient feature of Xarray. 

Let's verify that the data was in fact reordered when we reindexed the latitude coordinate. We can check this by looking at the temperature data values of ```tx``` and ```tx_reordered``` at a single time, single longitude, and all latitudes 

In [ ]:
# look at order of tx values where latitude is ordered ascending
tx.sel(time='2020-01',lon=-89.5, method="nearest").data

In [ ]:
# look at order of tx values after reordering latitude descending
tx_reordered.sel(time='2020-01',lon=-89.5, method="nearest").data

## Integer-based Selection

There are multiple other ways we can select data from Xarray data structures. 

### .isel()

The first method we'll cover is ```.isel()```. This function allows us to use dimension labels with integer positions (as opposed to coordinate labels). For example, if we want to select the first time of a DataArray without having to know the label for that particular time we could do:

In [ ]:
# select with dim name and integer position
tx.isel(time=0)

Slices with ```.isel()``` work the same way. Use the dimension name and integer positions. We can get fancy and use ```slice(start, exclusive stop, step)``` to select every other month of the first year of data. And just like basic indexing in NumPy, the ending index of an integer slice in Xarray is exclusive.

In [ ]:
# slice of every other month of the first year
tx.isel(time=slice(0,12,2))

### Basic Indexing with Square Brackets like NumPy

While not necessarily recommended, the other way to select with integer positions is to use no labels at all. This is the same as NumPy basic indexing where the order of the integer indexers correspond to the order of dimensions of ```tx```. For example, to select the first time (first integer 0), first lat (second integer 0), first lon (third integer 0):

In [ ]:
# selecting without labels (NumPy basic indexing)
tx[0,0,0]

Now, let's select a slice in time. Remember, this type of indexing, just like NumPy, is exclusive of the ending index.

In [ ]:
# selecting without labels (NumPy basic indexing)
tx[0:12,0,0]

Of course, just like we learned in the NumPy lesson, the order of the dimensions matters when using this indexing method. If we want to slice times, we need to know that the time dimension is axis 0.

## Combining Label-based and Integer-based Selection

Xarray is very flexible when it comes to different ways to select data within the DataArray structure. We can even combine different selection methods by stringing them together. Let's look at how to select the first time by position and latitude/longitude by label.

In [ ]:
# combining .isel() and .sel()
tx.isel(time=0).sel(lat=slice(30.,30.5),lon=slice(-89.0,-88.5))

<div class="alert alert-info"> 
    
## Exercise 2: Label-based and Integer-based Selection

Use the variable ```tx``` to perform the following data selections:


A) Select all latitudes and longitudes for February 2022.

In [ ]:
# add your code here


<div class="alert alert-info"> 

B) Select the timeseries for the nearest point to 32.2981° N, 90.1806° W.

In [ ]:
# add your code here


<div class="alert alert-info"> 

C) Select data for all months in 2016.

In [ ]:
# add your code here


<div class="alert alert-info"> 

D) Use ```.isel()``` to select all data at the last time.

In [ ]:
# add your code here


<div class="alert alert-info"> 

E) Select all data between 30°N and 31°N, 89°W and 90°W.

In [ ]:
# add your code here


<div class="alert alert-info"> 

F) Select every other month of data from January 2000 to December 2009.

**Hint:** look at the documentation for the ```slice()``` function.

In [ ]:
# add your code here


# IV. Data Analysis and Visualization with Xarray

When it comes to basics like elementwise operations, broadcasting, comparison and logic operators, math functions, searching, counting, and handling missing values, Xarray works in much the same ways as NumPy, since NumPy arrays underlay the Xarray data structures. We'll focus next on data analysis and visualization examples using Xarray that include some of these basics instead of comprehensive coverage of basic topics since it would be very similar to what we learned in the NumPy lesson.

We'll look at how we can use Xarray functions and data structures for the following data analyses:
- calculating summary statistics in time at each lat/lon location on the grid
- creating True/False masks and searching/replacing data values
- calculating accurate spatial means
- calculating area-mean climatology (mean annual cycle of tmax for Mississippi)
- accurate resampling of monthly tmax to seasonal means
- calculating monthly anomalies (departures from the monthly longterm mean)

In [ ]:
# starting with a fresh version of tx from the data file
tx = xr.open_dataset('data/nclimgrid/nclimgrid_tmax_199401-202312.nc').tmax
tx

## Summary Statistics like Mean and Standard Deviation

The biggest difference between Xarray and NumPy with these functions is that Xarray functions take the dimension name (label) as the input parameter whereas NumPy required the ```axis=``` parameter. A big advantage to using the dimension names instead of axis numbers is that we don't need to know the order of dimensions in our array (i.e., it doesn't matter if the time dimension is axis 0 or axis 2, using the label "time" will always select the correct dimension). 

We won't cover all the available summary and aggregation functions available in Xarray, but if you're interested in learning more they can generally be found in the [Aggregation](https://docs.xarray.dev/en/stable/api.html#id6) or [Computation](https://docs.xarray.dev/en/stable/api.html#id5) sections for DataArray functions in the Xarray API reference.

In [ ]:
# Compute summary statistics
# input parameter is dimension name(s)
tx_mean = tx.mean('time')

In [ ]:
# Compute summary statistics
# input parameter is dimension name(s)
tx_std = tx.std('time')

Don't worry too much about the warning above. You'll get that warning with ```.std()``` and a few other functions if your data contains grids cells where all values are nan. This data does contain all-nan grid cells (beyond the MS state boundary) so we can expect this behavior. The result of the standard deviation of all nan will be nan though (nan is propagated) so we can ignore the warning.

The Xarray data structure function ```.plot()``` is a quick way to visualize your data and includes relevant metadata labels automatically. It is based on matplotlib.

In [ ]:
# fast but not pretty plot
tx_mean.plot()

Notice how the NaN values (white/tranparent grid cells outside of Mississippi) are propagated to the results of ```.mean()``` and ```.std()```. This behavior is the same with NumPy data structures and NumPy ```.mean()``` and ```.std()```. Also, having NaN in your DataArrays (as opposed to filling NaN with some other value) is often beneficial when plotting because Xarray usually plots NaN in white/transparent. We could change the color of NaN cells easily as well, which we will see below.

## Plotting with Cartopy 

Our data doesn't have a spatial reference that indicates its coordinate reference system. Usually when this happens and the data dimensions are latitude and longitude we can assume the CRS is EPSG:4326. **To make our plots look nicer we can plot EPSG:4326 data with Pseudo Plate Carree (EPSG:9825) using the cartopy and matplotlib packages. Pseudo Plate Carree is not actually a map projection but just a way to make a nicer depiction of data with latitude/longitude coordinates on a computer display.** Cartopy also has features like coastlines, state and country boundaries, lakes, rivers, and a few other features from the Natural Earth project.

The functions we'll use below to create a figure come from three different packages: Xarray (e.g., ```.plot()```), cartopy (e.g., ```ccrs.PlateCarree()```, ```cfeature.STATES```) and matplotlib (e.g., all the other functions used below). Therefore, to find more information about available plotting functions and parameters we may need to look in three different places (although matlplotlib will be the main library to search for help): [matplotlib API reference](https://matplotlib.org/stable/api/index.html), [xarray.DataArray.plot()](https://docs.xarray.dev/en/latest/generated/xarray.DataArray.plot.html), [cartopy API reference](https://scitools.org.uk/cartopy/docs/latest/reference/index.html). 

You can also find example plots on these pages: **[matplotlib plot examples](https://matplotlib.org/stable/gallery/index.html)**, **[Xarray plot examples](https://docs.xarray.dev/en/latest/user-guide/plotting.html)**, **[cartopy plot examples](https://scitools.org.uk/cartopy/docs/latest/gallery/index.html)**   

Matplotlib and the packages that work alongside it can get very complicated and confusing very quickly. It can be especially confusing as to exactly where the documentation for a given function is located or which package documentation you should look to for finding additional plotting functions to use. The key to understanding plotting with matplotlib is recognizing all the different types of objects that are produced by the various plotting functions. **The object type determines where you go to look for documentation.** 

The main objects we'll be working with when it comes to plotting in this notebook are the matplotlib Figure object and the matplotlib Axes object. You can think of the Figure object as the overall container that holds everything we want to plot. Any functions that act on a Figure object to modify the plot produced will be found in the [API Reference for the matplotlib.figure class](https://matplotlib.org/stable/api/figure_api.html#module-matplotlib.figure). You can think of Axes objects as the individual plots inside the overall Figure container. Any functions that act on an Axes object to modify the plot produced will be found in the [API Reference for the matplotlib.axes class](https://matplotlib.org/stable/api/axes_api.html).    

Below, we'll make a figure that includes two sub plots (Axes objects), where the left sub plot is mean tmax and the right sub plot is tmax standard deviation.

In [ ]:
# year values, programmatically, for use in plot titles
year_start, year_end = tx.time.isel(time=0).dt.year.data, tx.time.isel(time=-1).dt.year.data

# create the overall figure object
fig = plt.figure(figsize=(8,4))

# create two plots (axes objects ax1, ax2) within the figure
# left plot
ax1 = fig.add_subplot(121, projection=ccrs.PlateCarree()) # first subplot using Pseudo Plate Carree for display
# modifications to the axes object
ax1.set_extent([-92, -87, 28, 37], crs=ccrs.PlateCarree()) # lat lon extents
ax1.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') # Add state boundaries
ax1.set_xticks(ax1.get_xticks()) # show x-axis label, ticks, tick labels
ax1.set_yticks(ax1.get_yticks()) # show y-axis label, ticks, tick labels
# plot and title on the axes object
tx_mean.plot(ax=ax1,cbar_kwargs={'label':'degrees C'}) # what to plot on this axes object, .plot is from xarray
ax1.set_title(f'{year_start}-{year_end} mean of tmax', fontsize=12)

# right plot
ax2 = fig.add_subplot(122, projection=ccrs.PlateCarree()) # second subplot using Pseudo Plate Carree for display
# modifications to the axes object
ax2.set_extent([-92, -87, 28, 37], crs=ccrs.PlateCarree()) 
ax2.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') 
ax2.set_xticks(ax2.get_xticks()) 
ax2.set_yticks(ax2.get_yticks()) 
# change the color of NaN cells
cm = plt.get_cmap("viridis") # get the default colormap object
cm.set_bad("gray") # now change the NaN color
# plot and title on the axes object
tx_std.plot(ax=ax2,cmap=cm,cbar_kwargs={'label':'degrees C'}) # what to plot on this axes object, .plot is from xarray
ax2.set_title(f'{year_start}-{year_end} standard dev of tmax', fontsize=12)

# plot everything in the figure object
plt.show()

If we wanted to cut down on repetitive code we could put the variable information in a dictionary and create the plot with a loop.

In [ ]:
# dict of variable info
# keys will appear in plot titles, values are the variables to plot
var_info = {'mean of tmax':tx_mean,'standard deviation of tmax':tx_std}

# create main figure object
fig = plt.figure(figsize=(8,4))
nrows,ncols = 1,2

for i, (key,value) in enumerate(var_info.items()):
    ax = fig.add_subplot(nrows,ncols,i+1, projection=ccrs.PlateCarree()) 
    ax.set_extent([-92, -87, 28, 37], crs=ccrs.PlateCarree()) 
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') 
    ax.set_xticks(ax.get_xticks()) 
    ax.set_yticks(ax.get_yticks()) 

    value.plot(ax=ax,cbar_kwargs={'label':'degrees C'}) 
    ax.set_title(f'{year_start}-{year_end} {key}', fontsize=12)

# plot everything
plt.show()

<div class="alert alert-info"> 
    
### Exercise 3: Find the 30-year Maximum Tmax at Each Point in Space

Use the Xarray DataArray function ```.max()``` to find the maximum value of ```tx``` in time at every spatial grid cell and save the result to a new variable called ```tx_max``` (this is similar to how we found the mean and standard deviation in time at every spatial grid cell). 

A) 
Plot ```tx_max``` and make sure your figure has the following:
- a title
- Psuedo Plate Carree display and state boundaries from the cartopy package
- latitude and longitude tick labels
- a colorbar with a label that states the data units

</div>

In [ ]:
# add your code here


<div class="alert alert-info"> 

B) Can you figure out how to keep the lat/lon tick labels (the integer values) but turn off the axis labels (i.e., "latitude [degrees_north]" and "longitude [degrees_east]")?

</div>

In [ ]:
# add your code here


## Creating Masks and Searching/Replacing Data Values

Just like with NumPy, comparison operators evaluate elementwise and return a True/False result. Also, Xarray ```xr.where()``` operates similarly to NumPy ```np.where()```. 

We'll build two masks to demonstrate. The mask will be True where ```tx``` is at least 32 degrees C and False otherwise. We'll create two versions of the mask: one where the mask contains only True and False values and another where the mask also contains NaN for grid cells without data. 

In [ ]:
# True/False mask, tx nans (beyond MS boundary) not propagated
warm_mask = tx >= 32
print(warm_mask.sizes)

# True/False mask, tx nans (beyond MS boundary) propagated
warm_mask_nan = xr.where(np.isfinite(tx),warm_mask,np.nan)
print(warm_mask_nan.sizes)

In [ ]:
# plot both masks for a single time
itime = 5
var_info = {f"32C True/False mask {tx.time[itime].dt.strftime('%Y-%m-%d').data}":warm_mask,
            f"32C mask w/nan {tx.time[itime].dt.strftime('%Y-%m-%d').data}":warm_mask_nan}

fig = plt.figure(figsize=(8,4))
nrows,ncols = 1,2

for i,(key,value) in enumerate(var_info.items()):
    ax = fig.add_subplot(nrows,ncols,i+1, projection=ccrs.PlateCarree()) 
    ax.set_extent([-92, -87, 28, 37], crs=ccrs.PlateCarree()) 
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') 
    ax.set_xticks(ax.get_xticks()) 
    ax.set_yticks(ax.get_yticks()) 

    value.isel(time=itime).plot(ax=ax,cbar_kwargs={'label':''})
    ax.set_title(key)
plt.show()

<div class="alert alert-info"> 
    
### Exercise 4: Use xr.where() to Create a Mask

Create a mask called ```warm38c``` with values of 0, 1, and np.nan. The mask should contain the following:
- a value of 1 where tx is greater than or equal to 38
- a value of 0 where tx is less than 38
- a value of np.nan where tx is np.nan

Plot the result for August 2023. Make your plot look similar to the masks we plotted in the previous code cell. 

In [ ]:
# add your code here


In [ ]:
# add your code here


## Spatial Means

To calculate accurate spatial means we need to weight the data by the area of each grid cell. For data on regular latitude/longitude grids like we are working with here, the area of a grid cell is proportional to the cosine of the latitude. Therefore, we can use the cosine of latitude as a proxy for grid cell area when computing area-weighted averages.

Xarray has a convenient function to accurately apply grid weighting ```.weighted()```. More information and examples can be found in the Xarray Documentation: [Xarray User Guide](https://docs.xarray.dev/en/stable/user-guide/computation.html#weighted-array-reductions), [Xarray Examples](https://docs.xarray.dev/en/latest/examples/area_weighted_temperature.html), [Xarray Tutorial](https://tutorial.xarray.dev/fundamentals/03.4_weighted.html), and [Xarray API Reference](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.weighted.html#xarray.DataArray.weighted).

First we'll calculate a DataArray of weights as the cosine of latitude and then we'll apply the weights to the spatial dimensions of the DataArray, and calculate the spatial mean.

In [ ]:
# cosine latitude weights
lat_weights = np.cos(np.deg2rad(tx.lat))
lat_weights.sizes, lat_weights[0:5]

In [ ]:
# weighted mean of tx over mississippi
tx_ms_weighted = tx.weighted(lat_weights).mean(['lat','lon'])
tx_ms_weighted.sizes

We'll do a quick comparison with the unweighted spatial mean. For this particular data, the weighting doesn't make much of a difference due to the fact that we're working with a small area (Mississippi) that doesn't extend very far in the north-south direction (less than 5 degrees latitude). The figure below shows differences on the scale of hundredths or thousandths of a degree. Of course, if the area we were averaging was at the country, hemisphere, or global scale, the difference would be more significant. 

In [ ]:
# calculated unweighted spatial mean
tx_ms_unweighted = tx.mean(['lat','lon'])

# plot the difference weighted minus unweighted
(tx_ms_weighted - tx_ms_unweighted).plot(figsize=(10,2))

<div class="alert alert-info"> 
    
### Exercise 5: Calculate an Accurate Spatial Mean

For the ```tx``` data from 31-34N and 90-92W, at each time calculate the spatial mean of the data weighted by the cosine of latitude. 

Accomplish the task with the following steps: 
- select the appropriate subset of data from ```tx``` and save it to a separate variable called ```tx_subset```
- create the weights and save them to a variable called ```subset_weights```
- weight ```tx_subset``` with your weights and save to a variable called ```tx_subset_weighted```
- plot the ```tx_subset_weighted``` timeseries

In [ ]:
# add your code here


## Area-mean Climatology (Annual Cycle of Mean Temperature)

At each grid cell in space, we'll calculate the monthly longterm means, also known as the climatology. Then we'll average the results in space over Mississippi. The use of labels and specialized xarray functions makes this process much less error prone.

In [ ]:
# 12 monthly longterm means at each grid cell
climatology = tx.groupby('time.month').mean('time')

# area-mean with cosine latitude weighting
ms_clim = climatology.weighted(lat_weights).mean(['lat','lon'])
ms_clim

In [ ]:
# visualize
ms_clim.attrs['units'] = 'degrees_C' # assign units so they appear on the y axis label
xvals = ms_clim.month # force there to be a tick mark at each of the 12 months

# plot
ms_clim.plot(figsize=(3,3),xticks=xvals)
plt.title('Monthly mean tmax for MS')
plt.show()


Notice above that we did not first create a Figure object and then create Axes objects inside the figure. We simply used Xarray ```.plot()``` which uses matplotlib.pyplot to create an Axes object. When we plot this way, the functions we can use to modify the plot like ```plt.title()``` will be found on the [API Reference page for matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html).

## Resampling in Time: Seasonal Means

Xarray adds ```.season``` to the list of possible datetime object components (like ```.year```, ```.month``` etc). Xarray seasons are DJF, MAM, JJA, SON. Seasonal means can be calculated using ```.groupby()```. Note that the string ```'time.season'``` below is the equivalent of ```tx.time.dt.season```.

In [ ]:
# seasonal means
# simple method but not completely accurate due differences in days per month
seasonal_mean = tx.groupby('time.season').mean('time')
seasonal_mean

In [ ]:
# create main figure object
fig = plt.figure(figsize=(8,8))

# loop to create sub plot for each season
for i,season in enumerate(seasonal_mean.season.data):
    ax = fig.add_subplot(2,2,i+1, projection=ccrs.PlateCarree()) # subplots using Pseudo Plate Carree for display
    ax.set_extent([-92, -87, 29, 36], crs=ccrs.PlateCarree()) # map lon/lat extents
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') # Add state boundaries
    ax.set_xticks(ax.get_xticks()) # show x-axis label, ticks, tick labels
    ax.set_yticks(ax.get_yticks()) # show y-axis label, ticks, tick labels
    seasonal_mean.sel(season=season).plot(ax=ax, add_labels=False, cbar_kwargs={'label':seasonal_mean.attrs['units']}) # what to plot on this Axes object
    ax.set_title(f'{season} mean of tmax', fontsize=12)

plt.tight_layout() # avoid text overlaps
plt.show() # plot everything

If we want to get very accurate with the seasonal mean calculation, we should weight the calculation by the number of days in each month. First, we can get the number of days in each of the 360 months using the ```.dt``` accessor.

In [ ]:
# get the number of days in each month of the time dimension
month_length = tx.time.dt.days_in_month
print(len(month_length))
print(month_length[0:12].data)

Then we can calculate a weight for each month in the timeseries using groupby. For example, the weight for January 1994 would be equal to 31 days divided by 2707 days (the total number of days in the DJF season from 1994-2023). The weights are such that the sum of all the monthly weights for each season should equal 1.   

We can use ```.groupy()``` to group the month lengths by season and then divide each month length by the total number of days in that season (over the whole timeseries of 30 years).  

In [ ]:
# calculate a weight for each month based on days in month
weights = month_length.groupby('time.season') / month_length.groupby('time.season').sum()
weights

In [ ]:
# verify that weights sum to 1 for each season
weights.groupby('time.season').sum()

Now we can calculate the seasonal means accurately by using the weights. After applying the weights and grouping by season, we'll need to use ```.sum()``` instead of ```.mean()```. 

In [ ]:
# calculate accurate seasonal means by using weights
weighted_seasonal_mean = (tx*weights).groupby('time.season').sum('time')

Let's take a quick look at the difference between the seasonal means and weighted seasonal means that we just calculated.  We'll use a different colormap to plot the differences. Here is a link to all the available [matplotlib colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html).

In [ ]:
# create main figure object
fig = plt.figure(figsize=(8,8))

# loop to create sub plot for each season
for i,season in enumerate(seasonal_mean.season.data):
    ax = fig.add_subplot(2,2,i+1, projection=ccrs.PlateCarree()) 
    ax.set_extent([-92, -87, 29, 36], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
    ax.set_xticks(ax.get_xticks()) 
    ax.set_yticks(ax.get_yticks())

    # plot the difference map
    # using vmin and vmax with a diverging colormap so that the color divergence point (white) is at zero
    (weighted_seasonal_mean - seasonal_mean).sel(season=season)\
    .plot(ax=ax, vmin=-.03, vmax=.03, cmap = 'bwr', add_labels=False, cbar_kwargs={'label':seasonal_mean.attrs['units']})
    
    ax.set_title(f'weighted minus unweighted\nseasonal mean {season} tmax', fontsize=12)

plt.tight_layout()
plt.show()

We can see differences both seasonally and spatially between our unweighted and weighted calculations, although the differences are small at only hundredths or thousands of a degree.

If the seasons DJF,MAM,JJA,SON aren't what we need, we could calculate the weighted means for custom seasons by defining the seasons with an additional data array coordinate. We can then take advantage of those coordinate labels with ```.groupby()``` to calculate new weights and new weighted seasonal means.

In [ ]:
# reminder of what month_length contains
month_length

In [ ]:
# a function to define the seasons
def get_season(month):
    if month in [1, 2, 3]:
        return "JFM"
    elif month in [4, 5, 6]:
        return "AMJ"
    elif month in [7, 8, 9]:
        return "JAS"
    else:
        return "OND"    

# create a list of 3-letter season for each time
custom_season = [get_season(m) for m in month_length.time.dt.month.data]
custom_season[0:9]

In [ ]:
# turn list of 3-letter seasons into a data array with matching time coordinate
custom_season = xr.DataArray(custom_season, coords={'time':('time', month_length.time.data)})
custom_season


In [ ]:
# assign the seasons as a new coordinate
month_length.coords['season']=custom_season
month_length

In [ ]:
# calculate weights based on the new seasons
# here, the string 'season' in the groupby is equivalent to month_length.season
weights = month_length.groupby('season') / month_length.groupby('season').sum()

# make sure each season sums to 1
weights.groupby('season').sum()

In [ ]:
# calculate weighted seasonal means for JFM, AMJ, JAS, OND and propagate nan
weighted_seasonal_mean = (tx*weights).groupby(weights.season).sum('time',skipna=False)
weighted_seasonal_mean

<div class="alert alert-info"> 
    
### Exercise 6: Calculate Accurate Seasonal Means

In this exercise you'll practice creating custom seasons and calculating accurate seasonal means by copying the code above and modifying it for the seasons JFMAM, JJ, and ASOND. 

A)
In the first code cell below do the following:
- create a custom function called ```get_season``` to define the three seasons,
- create a list called ```custom_season``` of 3-letter seasons for each time in the ```month_length``` array
- convert the ```custom_season``` list into a 1D Xarray data array with the dimension 'time' and coordinate 'time' of datetime objects
- set the `season` coordinate of the ```month_length``` array to ```custom_season```
- print ```month_length``` to the screen

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
B) In the next code cell below do the following:
- calculate new weights from ```month_length``` based on the new seasons
- make sure the weights for each season sum to 1

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
C) Now, re-calculate ```weighted_seasonal_mean``` using the new weights, propagating any nan values.

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
D) In the next code cell below do the following:
- make a figure with 3 plots, one for each season
- use a figsize of (12,4)
- use 1 row and 3 columns of axes objects
- make the plots look similar to those we've been creating in this section (i.e., use cartopy, limit extents, plot state boundaries, etc)

In [ ]:
# add your code here


## Calculating Anomalies and Plotting with Matplotlib GridSpec

Another thing that Xarray makes easy is calculating anomalies, in this case meaning the monthly departure from the monthly longterm means (climatology). We've already calculated the monthly climatology earlier in the notebook. We'll use that here to calculate the monthly anomalies. 

In [ ]:
# reminder of what climatology contains
climatology

In [ ]:
# calculate monthly anomalies
anomalies = tx.groupby('time.month') - climatology
anomalies.sizes

Let's look at the timeseries of anomalies at the grid cells closest to the MSU campus in Starkville and the Stennis Space Center on the Mississippi coast. We'll use ```matplotlib.gridspec.GridSpec``` (which we imported at the beginning of the notebook) to create a figure with 4 Axes objects laid out nicely.

In [ ]:
# first look at how to set up Axes with GridSpec

# create main figure object
fig = plt.figure(figsize=(20,6),layout="tight")

# set up multiple Axes on a grid
# this is how we want to lay out our plots
gs = GridSpec(2,5, figure=fig) # grid divided into 2 rows, 5 columns
ax0 = fig.add_subplot(gs[0, 0])  # first row, first column
ax1 = fig.add_subplot(gs[0, 1:]) # first row, remaining columns
ax2 = fig.add_subplot(gs[1, 0])  # second row, first column
ax3 = fig.add_subplot(gs[1, 1:]) # second row, remaining columns

# add modifications/formatting to all the axes
def format_axes(fig):
    for i, ax in enumerate(fig.axes):
        # formatting for the map plots
        if i in [0,2]:
            ax.text(0.5, 0.5, "map of point location", va="center", ha="center",fontsize=20)
            ax.text(0.5, 0.4, f"ax{i}", va="center", ha="center",fontsize=20)
            ax.tick_params(labelbottom=False, labelleft=False)
        # formatting for the timeseries plots
        if i in [1,3]:
            ax.text(0.5, 0.5, "anomaly timeseries", va="center", ha="center",fontsize=20)
            ax.text(0.5, 0.4, f"ax{i}", va="center", ha="center",fontsize=20)
            ax.tick_params(labelbottom=False, labelleft=False)

format_axes(fig)            

# plot everything
plt.show()

In [ ]:
# lat/lon of Starkville and Stennis
lat_stark, lon_stark = 33.4504, -88.8184
lat_stennis, lon_stennis = 30.3620, -89.5994

In [ ]:
# taking the code from above and adding in the data to plot on each axes object

# create main figure object
fig = plt.figure(figsize=(20,6),layout="tight")

# set up multiple axes on a grid
gs = GridSpec(2,5, figure=fig)
ax0 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax1 = fig.add_subplot(gs[0, 1:])
ax2 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[1, 1:])

# get nearest starkville lat lon in the data
nearest_lat = anomalies.sel(lat=lat_stark,lon=lon_stark,method='nearest').lat.data
nearest_lon = anomalies.sel(lat=lat_stark,lon=lon_stark,method='nearest').lon.data

# plot starkville point on axes 0
ax0.scatter(nearest_lon, nearest_lat, color='magenta', transform=ccrs.PlateCarree())
ax0.set_title('Starkville location')

# plot starkville timeseries on axes 1
anomalies.sel(lat=nearest_lat,lon=nearest_lon).plot(ax=ax1)
ax1.set_title('Starkville tmax anomalies', fontsize=12)

# get nearest stennis lat lon in the data
nearest_lat = anomalies.sel(lat=lat_stennis,lon=lon_stennis,method='nearest').lat.data
nearest_lon = anomalies.sel(lat=lat_stennis,lon=lon_stennis,method='nearest').lon.data

# plot Stennis location on axes 2
ax2.scatter(nearest_lon, nearest_lat, color='magenta', transform=ccrs.PlateCarree())
ax2.set_title('Stennis location')

# plot Stennis timeseries on axes 3
anomalies.sel(lat=nearest_lat,lon=nearest_lon).plot(ax=ax3)
ax3.set_title('Stennis tmax anomalies', fontsize=12)

# add modifications/formatting to all the axes
def format_axes(fig):
    for i, ax in enumerate(fig.axes):
        # formatting for the map plots
        if i in [0,2]:
            ax.set_extent([-92, -87, 30, 36], crs=ccrs.PlateCarree()) # map lat lon extents
            ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black') # add state boundaries
            ax.set_xticks(ax.get_xticks()) # show x-axis label, ticks, tick labels
            ax.set_yticks(ax.get_yticks()) # show y-axis label, ticks, tick labels
        # formatting for the timeseries plots
        if i in [1,3]:
            ax.axhline(0,color='grey',linestyle='dashed') # horizontal line at zero
            ax.set_ylabel('degrees C')

format_axes(fig)            

# plot everything
plt.show()


<div class="alert alert-info"> 
    
### Exercise 7: Use Gridspec to Plot Anomalies for Jackson, MS

Jackson, Mississippi is located at 32.2981° N, 90.1806° W. Using this location, create a similar figure to what we did above using Gridspec to make a figure with two plots: a map showing the closest lat/lon grid point on the left and the timeseries of anomalies for the closest lat/lon grid cell on the right. 

In [ ]:
lat_jackson, lon_jackson = 32.2981, -90.1806

# add your code here


# V. Other Useful Xarray Things

## Sorting

Sorting is different enough between NumPy and Xarray that it warrants demonstration. Xarray ```.sortby()``` makes sorting related data very easy, for example, sorting a data variable by it's coordinate values. We'll repeat the sorting we did in the NumPy lesson on the same data except in an Xarray DataArray structure.

First thing we'll do is build the Xarray DataArray. Don't worry too much about this step for now. We'll cover creating Xarray objects from Pandas and NumPy objects in detail in the next section.

In [ ]:
# create xarray data object from numpy ndarrays

# same data from the numpy lesson
t_mon = np.array([
 [39.02, 46.58, 53.02, 67.46, 77.70, 87.05, 91.73, 86.88, 78.14, 68.03, 52.94, 42.55],  # Las Vegas, station 100441
 [32.87, 33.24, 44.52, 52.60, 63.07, 77.38, 86.79, 80.60, 67.25, 60.35, 45.64, 32.50],  # Denver, station 100550
 [25.85, 29.41, 40.77, 48.34, 61.48, 71.98, 80.74, 73.79, 68.32, 53.36, 39.11, 31.58],  # Boston, station 100105
 [32.69, 37.63, 47.02, 52.47, 66.94, 77.12, 78.10, 77.97, 68.91, 56.97, 46.16, 35.59],  # NYC, station 100489
 [39.49, 44.85, 49.80, 56.81, 65.21, 74.73, 80.95, 75.09, 69.36, 60.90, 50.33, 47.40],  # Sacramento, station 100472
 [21.18, 24.31, 37.95, 50.16, 61.20, 72.38, 76.71, 76.00, 65.36, 54.25, 39.67, 30.56],  # Chicago, station 100872
 [47.39, 53.20, 52.34, 60.93, 60.49, 66.80, 68.82, 67.85, 65.02, 57.91, 52.14, 49.91],  # San Francisco, station 100514
 [37.13, 43.00, 52.34, 61.52, 72.23, 77.02, 83.13, 78.90, 74.37, 62.47, 52.82, 38.80],  # Charlotte, station 100111
 [57.68, 61.94, 65.26, 65.59, 74.52, 74.31, 75.95, 77.25, 77.32, 68.11, 63.24, 59.42],  # Los Angeles, station 100546
 [51.87, 57.80, 65.26, 73.61, 80.79, 86.59, 88.84, 89.54, 81.14, 75.56, 65.68, 54.30]])  # Austin, station 100646

station_number = np.array([100441, 100550, 100105, 100489, 100472, 100872, 100514, 100111, 100546, 100646])
station_lat = np.array([36.16, 39.74, 42.36, 40.71, 38.58, 41.88, 37.77, 35.23, 34.05, 30.27])
station_lon = np.array([-115.15, -104.99, -71.06, -74.01, -121.49, -87.63, -122.42, -80.84, -118.25, -97.74])
station_elev = np.array([ 610, 1609, 43, 10, 9, 181, 16, 229, 71, 149])
place_name = ['Las Vegas', 'Denver', 'Boston', 'NYC', 'Sacramento', 'Chicago', 'San Francisco', 'Charlotte', 'Los Angeles', 'Austin']

# create data array structure
t_mon = xr.DataArray(t_mon, 
                    coords = {'place_name' : ('place_name', place_name), 
                              'month' : ('month', np.arange(1,13))},
                    attrs = {'standard_name':'temperature',
                             'long_name':'longterm monthly mean temperature',
                             'units':'degree_F'})

# add additional coordinate info to the data array structure
t_mon = t_mon.assign_coords({'lat' : ('place_name', station_lat),
                     'lon' : ('place_name', station_lon),
                     'elev' : ('place_name', station_elev),
                     'station' : ('place_name', station_number)})
t_mon

In [ ]:
# sort data by increasing latitude
t_mon.sortby(t_mon.lat)

Notice how the data values and also all the coordinate values are sorted according to ascending latitude. With NumPy we had to use ```np.argsort()``` and advanced indexing. With Xarray, this task is much simpler.

The next three code cells are the Xarray equivalent to Exercise 9 in the NumPy lesson.

In [ ]:
# Sort the Denver row of the t_mon array

# Answer for NumPy array from NumPy Ex 9
# np.sort(t_mon[1,:])

# Answer for Xarray DataArray
denver_data = t_mon.sel(place_name='Denver')
denver_data.sortby(denver_data)

In [ ]:
# Sort all rows of the t_mon array by ascending station elevation.

# Answer for NumPy array from NumPy Ex 9
# sorted_elev = np.argsort(station_elev)
# t_mon[sorted_elev]

# Answer for Xarray DataArray
t_mon.sortby(t_mon.elev)

In [ ]:
# Sort all rows of the t_mon array by descending station elevation

# Answer for NumPy array from NumPy Ex 9
# sorted_elev = np.argsort(station_elev)[::-1]
# t_mon[sorted_elev]

# Answer for Xarray DataArray
t_mon.sortby(t_mon.elev,ascending=False)

## Xarray Assert Functions

Similar to NumPy and Pandas, Xarray offers a number of assert functions to test for differences between data in Xarray data structures. The full list of available functions is documented in the [Xarray API Reference in the Testing section](https://docs.xarray.dev/en/stable/api.html#testing).

The equivalent of ```np.testing.assert_allclose()``` in Xarray is ```xr.testing.assert_allclose()```.

The equivalent of ```np.testing.assert_array_equal()``` in Xarray is ```xr.testing.assert_equal()```.

A big difference with the two functions above is the Xarray functions will test if data values, dimensions, and coordinates are equal (but not names or attributes).

Xarray also offers the function ```xr.testing.assert_identical()``` that tests if *everything* is equal between two data structures including data values, dimensions, coordinates, names, and attributes.

We'll make a copy of ```t_mon``` and add one coordinate attribute to demonstrate.


In [ ]:
# copy t_mon and add one coordinate attribute

another_t_mon = t_mon.copy(deep=True)
another_t_mon.place_name.attrs['description'] = 'string city name of the station location'

Notice the Xarray ```.copy()``` function, used to make a deep copy of the DataArray.

In [ ]:
# assert data, dimensions, and coordiantes are equal (not names or attributes)
xr.testing.assert_equal(t_mon,another_t_mon)

In [ ]:
# assert everything is identical, including names and attributes
xr.testing.assert_identical(t_mon,another_t_mon)

# VI. Converting NumPy and Pandas Data Structures to Xarray Data Structures

Your data doesn't need to be provided in netcdf format in order to use Xarray data structures. You can hold any numerical data array in an Xarray DataArray or Dataset and add  metadata (labels) for the dimensions, coordinates, and attributes into the DataArray or Dataset object.

For example, you may have data in csv files that you want to get into Xarray data structures for the ease and convenience of analysis using xarray functions and labeled array capabilities. Luckily, it's fairly easy to convert NumPy and Pandas data structures into Xarray data structures. This section shows an example of NumPy to Xarray and Pandas to Xarray data structure conversion.

## NumPy to Xarray

The instructor has written the nclimgrid tmax data for 2023 over Mississippi to 12 separate csv files that have no headings or index labels at ```data/nclimgrid/nclimgrid_tmax_MS_YYYY-MM.csv```. The latitudes and longitudes associated with this data are in separate files at ```data/nclimgrid/nclimgrid_tmax_MS_latitude.csv```, ```data/nclimgrid/nclimgrid_tmax_MS_longitude.csv```. The time associated with each file is in the file name. This is the same nclimgrid tmax data we have been working with already, just now in csv files with separate metadata as opposed to in one tidy netcdf file.

Since there are no row or column labels in these files, we'll read the files into NumPy arrays and then convert to an Xarray DataArray.

First, we'll programmatically get the list of filenames.

In [ ]:
# get lists of files
data_files = glob.glob('data/nclimgrid/nclimgrid_tmax_MS_????-??.csv')
grid_files = glob.glob('data/nclimgrid/nclimgrid_tmax_MS_l*.csv')
data_files,grid_files

Next, we'll loop through each data file, reading the data into a NumPy array, and appending the array to a list object.

In [ ]:
# loop through files, appending numpy arrays to a list
np_arrays = []
for f in data_files:
    np_arrays.append(np.genfromtxt(f,delimiter=',',dtype='float32'))

# see shape of each array in list
for arr in np_arrays:
    print(arr.shape)

To build an Xarray data structure from this data we'll need information about the array dimensions (time, latitude, and longitude) to create the coordinates in the data structure. First we'll read in the latitudes and longitudes.

In [ ]:
# get lats and lons
lat = np.genfromtxt(grid_files[0],delimiter=',',dtype='float32')
lon = np.genfromtxt(grid_files[1],delimiter=',',dtype='float32')

# check that shapes match dimensions of the data arrays
lat.shape,lon.shape

Next, we'll create datetime objects for the time coordinate. The Python function ```.split()``` splits string objects into to parts based on a particular character. We use this function to get the first and last time string from the filenames and then input those strings into the ```xr.date_range()``` function, which is the same as the ```pd.date_range()``` function that we learned in the Pandas lesson.

In [ ]:
# look at the first data file path
print(data_files[0])

The first line of code below means: take the first string in the ```data_files``` list, split it into parts at the underscore character, from those parts take the last one which would be ```2023-01.csv```, from that get characters 0 through 6 which would be ```2023-01```, and set that equal to the variable called ```time_start```.

In [ ]:
# get first time and last time string from file names
time_start = data_files[0].split('_')[-1][0:7]
time_end = data_files[-1].split('_')[-1][0:7]
print(time_start,time_end)

# create monthly datetimes
time = xr.date_range(time_start,time_end,freq='MS')
time

Now that we have a list of NumPy arrays as well as coordinate information for time, latitude, and longitude, we can build an Xarray DataArray stucture with the ```xr.DataArray()``` function.

In [ ]:
# construct an Xarray data array structure from list of numpy arrays
tmax = xr.DataArray(np_arrays,
                    name='tmax',
                    coords = {'time':('time',time),
                              'lat':('lat',lat),
                              'lon':('lon',lon)})

# define coordinate metadata with dictionaries
time_attrs = {'standard_name':'time', 'long_name':'time, in monthly increments'}
lat_attrs = {'standard_name':'latitude', 'units':'degrees_north'}
lon_attrs = {'standard_name':'longitude', 'units':'degrees_east'}
var_attrs = {'standard_name':'air_temperature',
             'long_name':'Temperature, monthly average of daily maximums',
             'units':'degrees_C'}

# attach the metadata
tmax.time.attrs = time_attrs
tmax.lat.attrs = lat_attrs
tmax.lon.attrs = lon_attrs
tmax.attrs = var_attrs

tmax

## Pandas to Xarray

The instructor has written the nclimgrid tmax data for 2023 over Mississippi to 12 separate csv files *with* column headings (longitude grid values) and index labels (latitude grid values) at ```data/nclimgrid/nclimgrid_tmax_MS_YYYY-MM_with_labels.csv```. The time associated with each file is in the file name. This is the same nclimgrid tmax data we have been working with already, just now in csv files that have headings and index labels as opposed to in one tidy netcdf file.

Since there *are* row and column labels present in these files, we'll read the files into Pandas DataFrames and then convert to an Xarray DataArray.

We'll start once again by getting a list of file names.

In [ ]:
# get lists of files
data_files = glob.glob('data/nclimgrid/nclimgrid_tmax_MS_????-??_with_labels.csv')
data_files

We do a similar looped read except this time we're using ```pd.read_csv()``` and appending Pandas DataFrames to a list.

In [ ]:
# loop through files, appending pandas dataframes to a list
pd_frames = []
for f in data_files:
    pd_frames.append(pd.read_csv(f, index_col='lat',dtype='float32'))

# see shape of each dataframe in list
for frame in pd_frames:
    print(arr.shape)

All of the files contain the same latitude and longitude information, so we can pull that info from one of the DataFrames we read. The latitudes are the index values and the longitudes are the column names.

In [ ]:
# take a look at a single dataframe 
pd_frames[0]

In [ ]:
# get lats and lons
lat = pd_frames[0].index
lon = pd_frames[0].columns.astype('float32')

# see if shapes match dimensions of the dataframes
lat.shape,lon.shape

Again, we'll create datetime objects from the file names. These data file names are slightly different, so the code to get the first and last time string is slightly different.

In [ ]:
# take a look at a single file path
data_files[0]

In [ ]:
# get first time and last time string from file names
time_start = data_files[0].split('_')[-3]
time_end = data_files[-1].split('_')[-3]
print(time_start,time_end)

# create monthly datetimes 
time = xr.date_range(time_start,time_end,freq='MS')
time

Creating an Xarray DataArray structure uses the same code as before. The only exception is this time we are inputting a list Pandas DataFrames to ```xr.DataArray()``` instead of a list of NumPy arrays. Everything else is exactly the same.

In [ ]:
# this part is the same! (except now inputting a list of dataframes)
# construct an Xarray data array structure from list of pandas dataframes
tmax = xr.DataArray(pd_frames,
                    name='tmax',
                    coords = {'time':('time',time),
                              'lat':('lat',lat),
                              'lon':('lon',lon)})

# define coordinate metadata with dictionaries
time_attrs = {'standard_name':'time', 'long_name':'time, in monthly increments'}
lat_attrs = {'standard_name':'latitude', 'units':'degrees_north'}
lon_attrs = {'standard_name':'longitude', 'units':'degrees_east'}
var_attrs = {'standard_name':'air_temperature',
             'long_name':'Temperature, monthly average of daily maximums',
             'units':'degrees_C'}

# attach the metadata
tmax.time.attrs = time_attrs
tmax.lat.attrs = lat_attrs
tmax.lon.attrs = lon_attrs
tmax.attrs = var_attrs

tmax


<div class="alert alert-info"> 
    
## Exercise 8: Convert Numpy Arrays to Xarray DataArray

You have three 2D NumPy arrays, each array representing daily accumulated precipitation in mm with dimensions lat, lon for a single time. Use the information given and the NumPy arrays below to create a 3D Xarray DataArray that has a named data variable, named dimensions, coordinates for all dimensions, coordinate attributes, and variable attributes.

Metadata info:
- variable name: pr
- dimesion names: time, lat, lon
- time coordinate: 5 Aug 2020 to 7 Aug 2020
- lat coordinate: 30.0 to 34.0, evenly spaced every 1 degree
- lon coordinate: -91.5 to -88.5, evenly spaced every 0.5 degree
- time, lat, lon coordinate attributes and variable attributes: copy ```time_attrs```,```lat_attrs```,```lon_attrs```,```var_attrs``` from above. Keep all the dictionary keys but amend the values as you see appropriate.

In [ ]:
# NumPy array for 5 Aug 2020
arr1 =  np.array([[38, 0, 0, 0, 0, 0, 10],
                    [0, 0, 48, 0, 0, 0, 0],
                    [15, 0, 0, 0, 0, 0, 0],
                    [0, 29, 0, 11, 0, 0, 0],
                    [33, 35, 0, 0, 0, 28, 7]], dtype='int16')

# NumPy array for 6 Aug 2020
arr2 =  np.array([[18, 0, 0, 0, 16, 0, 0],
                [0, 0, 0, 0, 49, 0, 0],
                [50, 0, 0, 0, 0, 17, 0],
                [0, 0, 0, 0, 0, 13, 38],
                [23, 0, 0, 43, 0, 0, 0]], dtype='int16')

# NumPy array for 7 Aug 2020
arr3 =  np.array([[0, 0, 0, 0, 0, 0, 50],
                 [0, 45, 0, 0, 0, 0, 0],
                 [0, 0, 0, 32, 18, 0, 27],
                 [0, 11, 0, 0, 0, 0, 41],
                 [0, 0, 0, 5, 0, 42, 0]], dtype='int16')

<div class="alert alert-info"> 
    
A) First, create a DatetimeIndex of the three dates and save it to a variable called ```time```.

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
B) Next, programmatically create a two lists of floats, one for the lats and one for the lons. Save them to variables called ```lat``` and ```lon``` with the data type float32. Print your lists to the screen to verify they are correct. 

**Hint:** find the appropriate function to use by looking back to the NumPy notebook in the section *Array Creation Functions*. 

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
C) Now, set up the attribute dictionaries for ```time_attrs```,```lat_attrs```,```lon_attrs```,```var_attrs```.

In [ ]:
# add your code here


<div class="alert alert-info"> 
    
D) Finally, create an Xarray data array called ```pr``` with all the appropriate metadata attached. Print pr to the screen to verify everything worked correctly.

In [ ]:
# add your code here


# VII. Converting Xarray Data Structures to NumPy and Pandas

## Xarray to NumPy

We covered briefly at the beginning of the notebook how to convert data from an Xarray DataArray to a NumPy ndarray using ```.data```. Another function that does the same thing is ```.to_numpy()```.

In [ ]:
# xarray to numpy data structure

test = tmax.data
print(test.shape, type(test), test.dtype)

test = tmax.to_numpy()
print(test.shape, type(test), test.dtype)

We can, of course, also convert the coordinates into NumPy arrays as well using ```.data``` or ```.to_numpy()```.

In [ ]:
# xarray to numpy data structure

test = tmax.lat.data
print(test.shape, type(test), test.dtype)

test = tmax.time.data
print(test.shape, type(test), test.dtype)

## Xarray to Pandas

Coverting Xarray data to Pandas works best for 1D and 2D data since Pandas is designed for working with tabular data. Converting higher dimensional Xarray DataArrays to Pandas will result in MultiIndexed DataFrames, which we have not covered. MultiIndexed DataFrames are a bit difficult to work with and are definitely not as efficient as multidimensional NumPy or Xarray data structures. We won't cover the details of Pandas MultiIndexes but if you're interested in learning more please follow this link to [MultiIndex/Advanced Indexing in the Pandas User Guide](https://pandas.pydata.org/docs/user_guide/advanced.html).

1D data in Xarray data structures can be converted to Pandas Index objects, series, or dataframes with the ```.to_index()```, ```.to_pandas()```, and ```.to_dataframe()``` functions.

In [ ]:
# convert 1D Xarray data to Pandas index
test = tmax.time.to_index()
print(test.shape, type(test))
test

In [ ]:
# convert 1D Xarray data to Pandas series
test = tmax.time.to_pandas()
print(test.shape, type(test))
test.head()

In [ ]:
# convert 1D Xarray data to Pandas dataframe
test = tmax.time.to_dataframe()
print(test.shape, type(test))
test.head()

2D Xarray DataArrays can be converted to Pandas DataFrames with index and column labels automatically coming from the DataArray coordinate information using the ```.to_pandas()``` function. This function works on either 1D Xarray objects and returns a Pandas series (like we saw above) or a 2D Xarray object and returns a Pandas DataFrame. To demonstrate ```.to_pandas()``` on 2D Xarray data, we'll use the tmax DataArray at one single time.

In [ ]:
# convert 2D xarray data array to pandas dataframe
test = tmax.isel(time=0).to_pandas()
print(test.shape, type(test))
test.head()

Converting that same data to Pandas using ```.to_dataframe()``` results in a 2-level MultiIndexed dataframe where the MultiIndex contains the latitudes and longitudes, the single datetime is repeated for each row in a time column, and all the values of tmax end up in a single column.

In [ ]:
# convert 2D xarray data array to multiindexed pandas dataframe
test = tmax.isel(time=0).to_dataframe()
print(test.shape, type(test))
test.head()

For 3D data, the only function that can be used to convert to Pandas is ```.to_dataframe()``` (```.to_pandas()``` will error since it only accepts 1D and 2D inputs). The result will be a Pandas dataframe with a 3-level MultiIndex. The following code is just a demonstration however, working with 3D or higher dimensional data in this format is not advisable.

In [ ]:
test = tmax.to_dataframe()
print(test.shape, type(test))
test.head()

<div class="alert alert-info"> 
    
## Exercise 9: Convert Xarray DataArray Components to NumPy and Pandas

Using the ```pr``` DataArray variable you created in the last exercise, complete the following data structure conversion tasks.

A) First, convert ```pr``` to a 3D NumPy array and save it as ```pr_numpy```.

In [ ]:
# add your code here


<div class="alert alert-info"> 

B) Now, convert the ```pr``` data for 6 Aug 2020 to a Pandas Dataframe where the index values are the latitudes and the column names are the longitudes. Save the result to a new variable called ```pr_df```.

In [ ]:
# add your code here


<div class="alert alert-info"> 

C) Last, convert the longitude values from the ```pr``` DataArray to a Pandas Index and save the result to a new variable called ```lon_index```.

In [ ]:
# add your code here


# VIII. Estimating Memory Usage of Xarray Objects

With the ```.nbytes``` property, we can get information about the memory consumption of the underyling NumPy arrays within an Xarray DataArray structure. What if we want to know how much memory is consumed by the entire ```tx``` DataArray structure? Because there are so many components of an Xarray DataArray structure, we would need the ```.nbytes``` of the underlying NumPy array of temperature data, the ```.nbytes``` of the underlying NumPy arrays for all the coordinates, and the ```pd.Index.memory_usage()``` of the underlying Pandas Indexes of all the coordinates. We would also need to know how much memory is consumed by all of the attributes and other labels. **There is no convenient function for estimating the memory consumption of an entire DataArray object.** The best we can do is sum the ```.nbytes``` of all the underlying NumPy arrays and ```.memory_usage()``` of the all the underlying Pandas indexes. But this shouldn't be too far off from the actual memory consumption of a DataArray object since generally, attributes and other labels take up a negligible amount of memory anyway. 

In [ ]:
BtoMB=1E6 # bytes to MB conversion

# MB of all the underlying NumPy and Pandas data structures
print('tx array',tx.nbytes/BtoMB,'MB')
print('time array',tx.time.nbytes/BtoMB,'MB')
print('lat array',tx.lat.nbytes/BtoMB,'MB')
print('lon array',tx.lon.nbytes/BtoMB,'MB')
print('time index',tx.indexes['time'].memory_usage()/BtoMB,'MB')
print('lat index',tx.indexes['lat'].memory_usage()/BtoMB,'MB')
print('lon index',tx.indexes['lon'].memory_usage()/BtoMB,'MB')

# sum in MB of underlying data structures 
array_MBs = (tx.nbytes + 2*tx.time.nbytes + 2*tx.lat.nbytes + 2*tx.lon.nbytes)/1E6
print('entire tx DataArray structure is approximately',array_MBs,'MB')

We can see that even the coordinate arrays and indexes in the ```tx``` DataArray only occupy a fraction of a megabyte. This may not be the case for variables with many more times, latitude, or longitudes, but the coordinates will always occupy much much less memory than the DataArray. So really, for small variables like our ```tx```, a good estimate of the total size of the complex DataArray structure is simply the size of the underlying NumPy array of data ```tx.nbytes```. 

# IX. Writing Netcdf Files with Xarray

We can write any Xarray Dataset or DataArray to a new netcdf file (although DataArray objects will be converted on the fly to Dataset objects during the write). Here we'll practice preparing a Dataset that includes two data variables with all the appropriate metadata and writing the Dataset to a netcdf file with Xarray ```.to_netcdf()```. The two data variables we'll include in the Dataset are ```tx``` and a new variable we'll create called ```tx_F``` which will be ```tx``` converted to units of Fahrenheit.

In [ ]:
# convert units
tx_F = tx*(9/5)+32
tx_F

In [ ]:
# copy metadata
tx_F.attrs = copy.deepcopy(tx.attrs) # copy all attributes from tx
tx_F

In [ ]:
# fix metadata
for key in ['references','valid_min','valid_max']: del tx_F.attrs[key] # delete the ones that don't apply to tx_F
tx_F.attrs['units']='degree_Fahrenheit'  # new units

tx_F

In [ ]:
# create the dataset to be written to file
ds_out = tx.to_dataset() # create dataset from tx variable
ds_out['tmax_F']=tx_F    # add tx_F to the dataset

# assign file attributes
file_attrs={'source_code':'07_IntroToXarray.ipynb'}
ds_out.attrs=file_attrs

ds_out

Now that our Dataset object is prepared we can choose a file name and write it to a netcdf file.

In [ ]:
# write to file
outfile='tmax_nclimgrid_testwrite.nc'
ds_out.to_netcdf(outfile)

We can read the file we just wrote to double check that everything worked correctly.

In [ ]:
# read our new file
ds = xr.open_dataset(outfile)
ds

<div class="alert alert-info"> 

# X. Exercise: Putting It All Together

This exercise is in a separate notebook ```07_XarrayExercise.ipynb``` so that you can more easily reference the content in multiple lessons if you need to by having files open side-by-side in Jupyter Lab. When you're ready, go ahead and open the exercise notebook to proceed.

# XI. At a Glance: Language Covered

The Xarray functionality that we covered at a glance...

## Xarray Functions

```xr.DataArray()```, ```xr.date_range()```, ```xr.open_dataset()```, ```xr.testing.assert_allclose()```, ```xr.testing.assert_equal()```, ```xr.testing.assert_identical()``` ```xr.where()```


## Xarray data structure (Dataset or DataArray) methods
```.assign_coords()```, ```.astype()```, ```.groupby()```, ```.isel()```, ```.isin()```, ```.item()```, ```.mean()```, ```.plot()```, ```.reindex()```, ```.resample()```, ```.sel()```, ```.sortby()```, ```.std()```, ```.sum()```, ```.to_dataframe()```, ```.to_dataset()```, ```.to_index()```, ```.to_netcdf()```, ```.to_numpy()```, ```.to_pandas()```

## Xarray data structure (Dataset or DataArray) attributes and properties

```.attrs```, ```.coords```, ```.data```, ```.data_vars```, ```.dims```,  ```.dt.days_in_month```, ```.dt.month```, ```.dt.season```, ```.dt.strftime```, ```.dt.year```, ```.dtype```, ```.indexes```, ```.name```, ```.nbytes```, ```.ndim```, ```.shape```, ```.size```, ```.sizes```  


## Matplotlib functions

[From matplotlib.pyplot](https://matplotlib.org/3.5.3/api/_as_gen/matplotlib.pyplot.html): ```plt.figure()```, ```plt.get_cmap()```, ```plt.show()```, ```plt.tight_layout()```, ```plt.title()```

[Matplotlib Figure object functions](https://matplotlib.org/stable/api/figure_api.html#module-matplotlib.figure):
```.add_subplot()```

[Matplotlib Axes object functions](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.html#matplotlib.axes.Axes):
```.add_feature()```, ```.axhline()```, ```.get_xticks()```, ```.get_yticks()```, ```.scatter()```, ```.set_extent()```, ```.set_title()```, ```.set_xticks()```, ```.set_ylabel()```, ```.set_yticks()```, ```.text()```, ```.tick_params()```

[Matplotlib Colormap object functions](https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.Colormap.html): ```.set_bad()```

[From matplotlib.gridspec](https://matplotlib.org/stable/api/gridspec_api.html): ```GridSpec```

## Functions from other packages

From Cartopy: ```cartopy.crs.PlateCarree()```, ```cartopy.feature.STATES```

From Copy: ```copy.deepcopy()```

From Glob: ```glob.glob()```

From NumPy: ```np.allclose()```, ```np.cos()```, ```np.deg2rad()```, ```np.genfromtxt()```, ```np.isfinite()```, ```np.nan```

From Pandas: ```pd.read_csv()```, ```pd.DataFrame.astype()```, ```pd.DataFrame.columns```, ```pd.DataFrame.index```, ```pd.Index.memory_usage()```  

From core Python: ```enumerate()```, ```slice()```, ```list.append()```, ```string.split()```

<div class="alert alert-success">

# XII. Learning More About Xarray

For more about Xarray, start on the [Xarray website](https://docs.xarray.dev/en/stable/) where you can find:

- the getting started doc, user guide, and API reference documentation
- Xarray's online tutorial Jupyter notebooks and videos https://tutorial.xarray.dev/intro.html
    - which also includes a sub-tutorial "Xarray in 45 minutes" https://tutorial.xarray.dev/overview/xarray-in-45-min


Other useful Xarray tutorials can be found at:
- Project Pythia
    - Xarray tutorial notebook https://foundations.projectpythia.org/core/xarray.html
    - Cookbooks Gallery (task/analysis related notebook content that use multiple Python packages including Xarray) https://cookbooks.projectpythia.org/
- Xarray tutorial notebooks from Ryan Abernathy (formerly of Columbia University)
    - Xarray Fundamentals — Earth and Environmental Data Science https://earth-env-data-science.github.io/lectures/xarray/xarray.html
    - Xarray Interpolation, Groupby, Resample, Rolling, and Coarsen — Earth and Environmental Data Science https://earth-env-data-science.github.io/lectures/xarray/xarray-part2.html

</div>